# SEC Esgar all data

In [3]:
#!/usr/bin/env python3
"""
edgar_download.py
-----------------
Download SEC EDGAR public-entity indices and consolidate them into a single
CSV with an `entity_type` column distinguishing operating companies from
mutual fund share classes.

Endpoints fetched:
  - https://www.sec.gov/files/company_tickers.json
        Operating companies with a registered ticker (CIK + ticker + name).
  - https://www.sec.gov/files/company_tickers_exchange.json
        Same companies enriched with exchange (NYSE, Nasdaq, etc.).
  - https://www.sec.gov/files/company_tickers_mf.json
        Mutual funds / ETFs (series + class structure).

Output (written to ./edgar_data/):
  - sec_edgar_entities.csv
      Columns: entity_type, cik, cik10, ticker, name, exchange,
               series_id, class_id

      entity_type = "company"     -> operating company (series_id/class_id empty)
      entity_type = "fund_class"  -> a single share class of a mutual fund/ETF
                                     (name/exchange empty in EDGAR's MF feed)

Requirements:
  pip install pandas requests

Notes:
  - SEC EDGAR requires a descriptive User-Agent or returns HTTP 403.
    Edit USER_AGENT below before running.
"""

from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
import requests

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

USER_AGENT = "Dmytro Tkachenko (University of Tulsa) dmytro@example.edu"

OUTPUT_DIR = Path("edgar_data")
OUTPUT_FILE = OUTPUT_DIR / "sec_edgar_entities.csv"

ENDPOINTS = {
    "company_tickers":          "https://www.sec.gov/files/company_tickers.json",
    "company_tickers_exchange": "https://www.sec.gov/files/company_tickers_exchange.json",
    "company_tickers_mf":       "https://www.sec.gov/files/company_tickers_mf.json",
}

REQUEST_DELAY_SEC = 0.2  # well under the 10 req/s limit

FINAL_COLUMNS = [
    "entity_type", "cik", "cik10", "ticker", "name",
    "exchange", "series_id", "class_id",
]


# ---------------------------------------------------------------------------
# Fetching
# ---------------------------------------------------------------------------

def fetch_json(url: str) -> dict:
    """GET a JSON resource from SEC EDGAR. requests handles gzip transparently."""
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
        "Host": "www.sec.gov",
    }
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()


# ---------------------------------------------------------------------------
# Parsers — each EDGAR file has a different shape
# ---------------------------------------------------------------------------

def parse_company_tickers(payload: dict) -> pd.DataFrame:
    """{"0": {"cik_str": 320193, "ticker": "AAPL", "title": "Apple Inc."}, ...}"""
    df = pd.DataFrame.from_dict(payload, orient="index")
    df = df.rename(columns={"cik_str": "cik", "title": "name"})
    df["cik"] = df["cik"].astype(int)
    return df[["cik", "ticker", "name"]].reset_index(drop=True)


def parse_company_tickers_exchange(payload: dict) -> pd.DataFrame:
    """{"fields": [...], "data": [[cik, name, ticker, exchange], ...]}"""
    df = pd.DataFrame(payload["data"], columns=payload["fields"])
    df["cik"] = df["cik"].astype(int)
    for col in ("ticker", "exchange", "name"):
        if col in df.columns:
            df[col] = df[col].fillna("")
    return df[["cik", "ticker", "name", "exchange"]].reset_index(drop=True)


def parse_company_tickers_mf(payload: dict) -> pd.DataFrame:
    """{"fields": [...], "data": [[cik, seriesId, classId, symbol], ...]}"""
    df = pd.DataFrame(payload["data"], columns=payload["fields"])
    df = df.rename(columns={
        "seriesId": "series_id",
        "classId": "class_id",
        "symbol": "ticker",
    })
    df["cik"] = df["cik"].astype(int)
    return df[["cik", "series_id", "class_id", "ticker"]].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Consolidation
# ---------------------------------------------------------------------------

def build_unified_dataframe(
    tickers_df: pd.DataFrame,
    exchange_df: pd.DataFrame,
    mf_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge the operating-companies file with the exchange enrichment,
    stack the mutual-fund classes underneath, and add `entity_type` +
    a padded `cik10` column.
    """
    # operating companies: outer-join the two files on (cik, ticker)
    companies = tickers_df.merge(
        exchange_df[["cik", "ticker", "exchange"]],
        on=["cik", "ticker"],
        how="outer",
    )
    # outer-join may introduce NaN names when a row is only in exchange_df;
    # fill those from exchange_df's own name column
    name_lookup = exchange_df.set_index(["cik", "ticker"])["name"]
    companies["name"] = companies.apply(
        lambda r: r["name"] if isinstance(r["name"], str) and r["name"]
        else name_lookup.get((r["cik"], r["ticker"]), ""),
        axis=1,
    )
    companies["exchange"] = companies["exchange"].fillna("")
    companies["entity_type"] = "company"
    companies["series_id"] = ""
    companies["class_id"] = ""

    # mutual fund share classes
    funds = mf_df.copy()
    funds["entity_type"] = "fund_class"
    funds["name"] = ""
    funds["exchange"] = ""

    unified = pd.concat([companies, funds], ignore_index=True)
    unified["cik10"] = unified["cik"].map(lambda c: f"{int(c):010d}")

    # tidy: sort, deduplicate exact duplicates, enforce column order
    unified = (unified[FINAL_COLUMNS]
               .drop_duplicates()
               .sort_values(["entity_type", "cik", "ticker"])
               .reset_index(drop=True))
    return unified


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    if "example.edu" in USER_AGENT:
        print("[WARN] Edit USER_AGENT at the top of the script with your real "
              "name and email before running — SEC requires this.\n")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Fetching SEC EDGAR indices...")
    payloads = {}
    for name, url in ENDPOINTS.items():
        print(f"  GET {url}")
        payloads[name] = fetch_json(url)
        time.sleep(REQUEST_DELAY_SEC)

    print("\nParsing...")
    tickers_df  = parse_company_tickers(payloads["company_tickers"])
    exchange_df = parse_company_tickers_exchange(payloads["company_tickers_exchange"])
    mf_df       = parse_company_tickers_mf(payloads["company_tickers_mf"])

    print("Consolidating...")
    unified = build_unified_dataframe(tickers_df, exchange_df, mf_df)
    unified.to_csv(OUTPUT_FILE, index=False)
    print(f"  wrote {len(unified):>7,d} rows -> {OUTPUT_FILE}")

    # summary
    print("\nSummary:")
    counts = unified["entity_type"].value_counts()
    for kind, n in counts.items():
        print(f"  {kind:<12s} {n:>7,d}")

    companies = unified[unified["entity_type"] == "company"]
    with_exch = (companies["exchange"] != "").sum()
    print(f"\n  companies with exchange known:  {with_exch:,} / {len(companies):,} "
          f"({with_exch / len(companies):.1%})")

    if (companies["exchange"] != "").any():
        print("\n  exchange distribution (companies):")
        ex = companies.loc[companies["exchange"] != "", "exchange"].value_counts()
        print(ex.to_string().replace("\n", "\n    "))


if __name__ == "__main__":
    main()

[WARN] Edit USER_AGENT at the top of the script with your real name and email before running — SEC requires this.

Fetching SEC EDGAR indices...
  GET https://www.sec.gov/files/company_tickers.json
  GET https://www.sec.gov/files/company_tickers_exchange.json
  GET https://www.sec.gov/files/company_tickers_mf.json

Parsing...
Consolidating...
  wrote  38,689 rows -> edgar_data/sec_edgar_entities.csv

Summary:
  fund_class    28,308
  company       10,381

  companies with exchange known:  10,150 / 10,381 (97.8%)

  exchange distribution (companies):
exchange
    Nasdaq    4276
    NYSE      3283
    OTC       2564
    CBOE        27
